# To Build a MTN MANAGEMENT SYSTEM 

In [ ]:
pip install pymysql

In [2]:
# To import the Libraries 
import pymysql 
import os 
import cryptography 

In [4]:
PASSWORD = os.environ.get("MYSQL_PASSWORD")

In [6]:
# To establish a connection between MySQL and Python.
try:
    database = pymysql.connect(
        host = "localhost",
        port = 3306,
        user = "root", 
        password = PASSWORD
    )
    cursor = database.cursor()
    if cursor:
        print("Connection Successful..")
except cursor.error as error:
    print(f"Error message:{error}")

Connection Successful..


In [8]:
# To display all the databases from mysql
cursor.execute("Show databases")
for i in cursor:
    print(i)

('bank',)
('car_dealer',)
('db15',)
('earlycode',)
('ec_mfb',)
('glovo',)
('hospital',)
('hotel',)
('inec',)
('information_schema',)
('items',)
('movie_rental',)
('mtn_db',)
('mysql',)
('performance_schema',)
('pop_sys',)
('portal',)
('sakila',)
('shoe_collections',)
('sys',)
('world',)


In [10]:
# To display all the columns in a table
cursor.execute("use mtn_db")
cursor.execute("show tables in mtn_db")
for i in cursor:
    print(i)

('plans',)
('subscribers',)


In [12]:
# To generate a Phone number for the user 
import random 
def generate_num():
    # To get the first four numbers.
    prefix = ["0813", "0903", "0703", "0803", "0706", "0806", "0906"]
    firstnum = random.choice(prefix)

    # To generate the rest.
    num = "".join([str(random.randint(0,9)) for _ in range(7)])
    return firstnum + num
generate_num()

'08031912366'

In [14]:
cursor.execute("show columns in subscribers")
for i in cursor:
    print(i)

('phone_no', 'varchar(11)', 'NO', 'PRI', None, '')
('Full_name', 'varchar(50)', 'NO', 'UNI', None, '')
('Gender', "enum('Male','Female')", 'NO', '', None, '')
('DOB', 'date', 'NO', '', None, '')
('REGISTRATION', 'timestamp', 'YES', '', 'CURRENT_TIMESTAMP', 'DEFAULT_GENERATED')
('ADDRESS', 'text', 'NO', '', None, '')


In [40]:
# The main Code 
from decimal import Decimal, InvalidOperation
import random 
import datetime as dt 
cursor.execute("use mtn_db")
def register():
    print("\n" + "+" * 50)
    print("Yello Welcome to the registration Section")
    full_name = input("Enter your full name: ")
    Gender = input("Enter your Gender(Male or Female): ")
    DOB_str = input("Enter your date of birth(dd/mm/yy): ")
    Address = input("Enter your place of Resident: ")

    def generate_num():
    # To get the first four numbers.
        prefix = ["0813", "0903", "0703", "0803", "0706", "0806", "0906"]
        firstnum = random.choice(prefix)
    
        # To generate the rest.
        num = "".join([str(random.randint(0,9)) for _ in range(7)])
        return firstnum + num
    phone_no = generate_num()
    DOB = dt.datetime.strptime(DOB_str, "%d/%m/%Y").date()
    query = "insert into subscribers(phone_no, Full_name, Gender, DOB, Address) values(%s, %s, %s, %s, %s)"
    cursor.execute(query,(phone_no, full_name, Gender, DOB, Address))
    database.commit()
    print(f"Registration successful, Your Phone number is {phone_no}")

def search():
    print("+" * 50)
    print("Yello Welcome to the Verification Section")
    phone_no = input("Enter your Phone number to confirm if you are user: ")
    query  = "select * from subscribers where phone_no = %s"
    try:
        cursor.execute(query, phone_no)
        result = cursor.fetchone()
        if result == 0:
            return f"Sorry, User not Found.."
        else:
            print(f"\n{result[0]}|{result[1]}|{result[2]}|{result[3]}|{result[4]}|{result[5]}")
    except Exception as e:
        print(f"Error message:{e}")
def Plans():
    print("\n" + "+" * 50)
    print("Yello Welcome to the Plans Section")
    print("1. >> Buy Airtime\n2. >> Buy Data")
    choice = input("Select from the following (1 or 2): ").strip()
    if choice == "1":  
        try:
            phone_no = input("Enter your phone number: ")
            amount_input = input("Enter Airtime Amount to purchase: ₦").strip()
            amount = Decimal(amount_input) # Safe exact currency math
            if amount <= 0:
                print("❌ Amount must be greater than zero.")
                return
            query = """INSERT INTO Plans (phone_no, plans_duration, price, AIRTIME, data_mb)
                VALUES (%s, 'Daily', %s, %s, 0.0)"""
            cursor.execute(query, (phone_no, amount, amount))
            database.commit()
            print("🎉 Airtime Purchase was successful!")

        except (ValueError, InvalidOperation):
            print("❌ Invalid input! Please enter a proper numeric amount.")
    elif choice == "2":
        # Formulate exact choice matching for your ENUM("Daily", "Weekly", "Monthly")
        print("\nSelect Duration:")
        print("D. Daily\nW. Weekly\nM. Monthly")
        phone_no = input("Enter your phone number: ")
        duration_input = input("Choice (D/W/M): ").strip().upper()
        
        # Explicit mapping ensures your database ENUM constraint never breaks
        duration_map = {"D": "Daily", "W": "Weekly", "M": "Monthly"}
        if duration_input not in duration_map:
            print("Invalid duration selected...")
            return
        plans_duration = duration_map[duration_input]

        try:
            data_input = input("How much Data (in MB): ").strip()
            data_mb = Decimal(data_input)
            simulated_price = (data_mb / 1000) * 100 

            if data_mb <= 0:
                print("Data amount must be greater than zero.")
                return
            # Insert structure matching your exact table setup
            query = """ INSERT INTO Plans (phone_no, plans_duration, price, AIRTIME, data_mb)
                VALUES (%s, %s, %s, 0.0, %s)"""
            cursor.execute(query, (phone_no, plans_duration, simulated_price, data_mb))
            database.commit()
            print(f"🎉 Data Purchase of {data_mb} MB was successful!")

        except (ValueError, InvalidOperation):
            print("❌ Invalid input! Data volume must be a number.")
    else:
        print("❌ Unknown selection. Please try again.")

def main():
    while True:
        print("\n"+"+" * 50)
        print("Yello Welcome to the MTN Menu Section")
        print("1. >> Register User \n2. >> Verify User\n3. >> To Buy Plans\n4. >> Logout")
        print("+" * 50)
        selection = input("Select from the following: ")
        if selection == "1":
            register()
        elif selection == "2":
            search()
        elif selection == "3":
             Plans()
        elif selection == "4":
            print("Thank you for using MTN App..")
            break
        else:
            print("Invaild Inputation.")

In [42]:
main()


++++++++++++++++++++++++++++++++++++++++++++++++++
Yello Welcome to the MTN Menu Section
1. >> Register User 
2. >> Verify User
3. >> To Buy Plans
4. >> Logout
++++++++++++++++++++++++++++++++++++++++++++++++++


Select from the following:  1



++++++++++++++++++++++++++++++++++++++++++++++++++
Yello Welcome to the registration Section


Enter your full name:  Ken Jefferson 
Enter your Gender(Male or Female):  Male 
Enter your date of birth(dd/mm/yy):  23/03/1967
Enter your place of Resident:  Asokoro 


Registration successful, Your Phone number is 07037376976

++++++++++++++++++++++++++++++++++++++++++++++++++
Yello Welcome to the MTN Menu Section
1. >> Register User 
2. >> Verify User
3. >> To Buy Plans
4. >> Logout
++++++++++++++++++++++++++++++++++++++++++++++++++


...

Select from the following:  2


++++++++++++++++++++++++++++++++++++++++++++++++++
Yello Welcome to the Verification Section


Enter your Phone number to confirm if you are user:  09031523463



09031523463|Temi Steven |Male|1996-05-01|2026-06-22 17:25:15|Maraba 

++++++++++++++++++++++++++++++++++++++++++++++++++
Yello Welcome to the MTN Menu Section
1. >> Register User 
2. >> Verify User
3. >> To Buy Plans
4. >> Logout
++++++++++++++++++++++++++++++++++++++++++++++++++


KeyboardInterrupt: Interrupted by user